# S04: mini project, a verifiable policy service

The notebook anchors the 90-minute build block inside a full three-hour mini project. Produce evidence that an answer is correct, authorised and supported. A source ID alone is inadequate.

In [ ]:
from pathlib import Path
import os, sys, json, tempfile
# Run from repository root, notebooks/, or a Colab clone.
candidates=[Path.cwd(), Path.cwd().parent, Path('/content/ai-platform-architect-labs')]
ROOT=next((p for p in candidates if (p/'engine').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open the extracted repository. Colab: follow README Setup first.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–20 min | Run the service and inspect evidence.** Ask the standard payment question, a restricted discount question and an unknown question. In rehearsal you get actual retrieved passages. Generated answers and semantic acceptance require live calls.

In [ ]:
from engine.evaluate import semantic_review
ix=Index();ix.build('v1',read_data('policies.json'));db=Purchases();app=Platform(ix,db,client)
for question in ['What are standard supplier payment terms?','What is the negotiated discount schedule?','What is the annual holiday allowance?']:
    print(question, '\n', json.dumps(app.ask(BUYER,'policy',question),indent=2))

**20–40 min | Demonstrate a misleading citation.** The answer below cites a real, exact quote but makes the opposite business claim. Observe the structural validator pass. The validator checks provenance syntax, not semantic entailment. Reviewers must examine the supported meaning.

In [ ]:
hits=ix.search('standard supplier payment terms',BUYER)
source=next(h for h in hits if h['id']=='A-TERMS')
misleading={'status':'answered','answer':'Standard suppliers are paid in 90 days.',
            'evidence':[{'id':source['id'],'quote':source['text']}]}
assert evidence_errors(misleading,hits)==[]
print('Structural errors:',evidence_errors(misleading,hits))
print('Human review: INCORRECT. The quoted source says 45 days, not 90.')

**40–65 min | Build the acceptance evidence.** Use nine cases including a partially answerable question. For every live answer, mark correct, supported and complete separately. Preserve failures; do not edit actual outputs to match labels. A refusal may be appropriate but must not invent a reason. `None` means unreviewed.

In [ ]:
review=semantic_review(app)
Path('outputs').mkdir(exist_ok=True)
Path('outputs/mini-project-review.json').write_text(json.dumps(review,indent=2))
print('Saved',len(review),'cases; human review remains pending.')
print(json.dumps(review[-1],indent=2))

**65–90 min | Implement one improvement.** Choose source-aware chunking, clearer missing-fact responses, or a permission-revocation design. Add two new tests before changing the code/prompt. Show the failed case, the change and the new result. Do not merely rerun the supplied cells.

**Team deliverable:** service demonstration; one architecture diagram; two new cases; nine reviewed answers if live; one limitation and one improvement. During the five 8-minute demos, use 3 minutes for behavior, 3 for failure evidence and 2 for questions. Assignment 1 turns this into an individual ADR. Close resources when finished.

In [ ]:
ix.close();db.close()